# Frozen Prithvi + WCE Warm-up → Direct DMI + Smooth Logit Morphology — Xuân Thủy

**Bản đồ mục tiêu: 21/09/2026.** Đây là một run mới, giữ nguyên sáu ảnh DN, sáu band, split, patch, batch schedule, offset −1000 và chuẩn hóa của notebook WCE.

Run này **không nạp checkpoint WCE cũ**. Nó bắt đầu từ Prithvi-EO-2.0-300M-TL pretrained gốc, đóng băng encoder, dùng WCE để khởi động neck/UNet/head, rồi chỉ chuyển sang exact DMI + morphology khi gate ổn định đạt hai lần liên tiếp. Mọi artifact nằm trong run folder P5 riêng; output WCE và direct-DMI P4 cũ không bị ghi đè.


### Thiết kế đã khóa

```text
Warm-up (312–936 step)
6 ảnh × 6 band → frozen Prithvi encoder (eval, no grad)
→ trainable learned neck + UNet decoder + 11-class head
→ raw logits → WCE
Morphology: frozen và bypass

Sau gate
frozen Prithvi encoder → task side + trainable smooth closing 3×3
→ Softmax → exact DMI, batch hiệu dụng 16
```

Readiness được kiểm tra mỗi 156 step: validation DMI đủ rank/hữu hạn; exact-DMI backward hữu hạn trên bốn batch train cố định; validation dự đoán đủ 11 lớp; recall lớp 5/6/7 đều ≥ 0,05. Chọn **checkpoint sớm nhất** đạt hai lần liên tiếp, không chọn theo WCE F1.

Giai đoạn DMI lưu hai model độc lập: macro-F1 validation là chính; validation DMI loss đủ rank là phụ. Encoder vẫn frozen trong nhánh P5 đầu tiên này; partial fine-tuning sẽ là thí nghiệm khác.


## 1. Kết nối Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Cài thư viện và helper theo commit cố định


In [ ]:
import subprocess,sys
from pathlib import Path

WORK=Path('/content/prithvi_xtseg_staged_workspace')
WORK.mkdir(parents=True,exist_ok=True)
requirements=['terratorch==1.2.12','torchgeo==0.9.0','timm==1.0.30',
              'huggingface-hub==1.33.0','rasterio==1.5.2','pandas==2.2.3',
              'numpy==2.3.5','einops==0.8.2','matplotlib>=3.8,<4',
              'PyYAML>=6,<7','geopandas>=1,<2','tqdm>=4.66,<5']
subprocess.run([sys.executable,'-m','pip','install','-q',*requirements],check=True)

BASELINE_COMMIT='0775770ae87b83b7d2982ed727226128685f3fba'
BASELINE=WORK/'public_baseline'
if not BASELINE.exists():
    subprocess.run(['git','clone','https://github.com/hongphuc-maker/xuan-thuy-dmi-segmentation.git',str(BASELINE)],check=True)
subprocess.run(['git','-C',str(BASELINE),'checkout','--detach',BASELINE_COMMIT],check=True)
head=subprocess.check_output(['git','-C',str(BASELINE),'rev-parse','HEAD'],text=True).strip()
assert head==BASELINE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(BASELINE)],check=True)
sys.path.insert(0,str(BASELINE/'src'))
print('Helper commit:',head)


## 3. Nạp payload staged experiment đã khóa SHA256


In [ ]:
import base64,hashlib,io,zipfile

PAYLOAD_B64='...'
PAYLOAD_SHA256='b2a19ceed5ea2a43a30dce054cc1d64b538e6207525d74f63ebc2a72fdde035a'
payload=base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest()==PAYLOAD_SHA256,'Payload checksum khác'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for entry in archive.infolist():
        destination=(WORK/entry.filename).resolve()
        if WORK.resolve() not in destination.parents:
            raise ValueError('Unsafe archive path')
    archive.extractall(WORK)

loaded=sys.modules.get('prithvi_xtseg')
if loaded is not None:
    loaded_path=Path(getattr(loaded,'__file__','')).resolve()
    if WORK.resolve() not in loaded_path.parents:
        for name in list(sys.modules):
            if name=='prithvi_xtseg' or name.startswith('prithvi_xtseg.'):
                del sys.modules[name]
sys.path.insert(0,str(WORK))
import torch,terratorch,importlib.metadata
print({p:importlib.metadata.version(p) for p in ['torch','torchvision','terratorch','torchgeo','timm','segmentation-models-pytorch','numpy']})
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — cần GPU để train')
print('Payload:',PAYLOAD_SHA256)


## 4. Cấu hình run P5 riêng


In [ ]:
import json,re

RUN_SUFFIX=''
DATA_ROOT=Path('/content/drive/MyDrive/Prithvi_Colab/data')
RUNS_ROOT=Path('/content/drive/MyDrive/Prithvi_Colab/runs')
DN_OFFSET=-1000.0
OFFSET_CONFIRMED=False
CLOUD_MASKS={}

assert re.fullmatch(r'[A-Za-z0-9_-]*',RUN_SUFFIX)
config=json.loads((WORK/'experiment_staged_warmup_dmi_morphology.json').read_text(encoding='utf-8'))
config['experiment_id']+=RUN_SUFFIX
config['radiometry']['dn_offset']=DN_OFFSET
config['radiometry']['confirmed_from_export_code']=OFFSET_CONFIRMED
config['cloud_masks']=CLOUD_MASKS
assert config['initialization']['WCE_checkpoint'] is None
assert config['warmup']['minimum_steps']==312
assert config['warmup']['maximum_steps']==936
assert config['warmup']['readiness_interval_steps']==156
assert config['warmup']['consecutive_ready_evaluations']==2
assert config['training']['encoder_policy']=='frozen'
assert config['objective']=={
    'name':'exact_dmi','matrix_jitter':0.0,
    'effective_batch_implementation':'two-pass microbatch recomputation with RNG replay'}
CONFIG_PATH=WORK/'effective_experiment.json'
CONFIG_PATH.write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
RUN_ROOT=RUNS_ROOT/config['experiment_id']
print('Run mới:',RUN_ROOT)
print('Ngày:',config['active_dates'],'→ mục tiêu',config['target_date'])
print('Warm-up:',config['warmup'])
print('Final selection:',config['selection'])


## 5. Audit dữ liệu và fixed train-only panel


In [ ]:
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
from prithvi_xtseg.training import open_experiment
from prithvi_xtseg.reporting import (show_inputs,show_split,show_history,
                                     show_warmup_history,show_map,show_selected_checkpoints)

experiment=open_experiment(CONFIG_PATH,DATA_ROOT,RUN_ROOT)
audit=json.loads((RUN_ROOT/'prepared'/'DATA_AUDIT.json').read_text())
panel=json.loads((RUN_ROOT/'FIXED_TRAIN_PANEL.json').read_text())
display(pd.DataFrame(audit['images']))
display(pd.DataFrame({'code':range(1,12),
                      'class':[config['class_map'][str(i)] for i in range(1,12)],
                      'train_pixels':audit['train_class_counts']}))
print({k:audit[k] for k in ['shape','common_labeled_pixels','train_pixels','validation_pixels',
                            'n_patches','n_batches','n_validation_windows','shared_input_pixels']})
print('Fixed train panel:',{k:panel[k] for k in ['n_batches','n_patches','used_for']})
assert audit['n_batches']==312
assert audit['shared_input_pixels']==0
fig=show_inputs(experiment);plt.show();plt.close(fig)
fig=show_split(experiment);plt.show();plt.close(fig)


## 6. Frozen-encoder WCE warm-up và readiness gate

Cell này bắt đầu từ Prithvi pretrained gốc khi run P5 chưa có resume. Nó tự resume `warmup_latest_resume.pt`, kiểm tra mỗi 156 step và dừng ở checkpoint sớm nhất đạt hai lần liên tiếp. Nếu tới step 936 vẫn không đạt, notebook dừng và ghi `WARMUP_BLOCKED.json`; không tự chuyển sang DMI.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime → Change runtime type → GPU rồi chạy lại')
warmup_result=experiment.warmup()
display(warmup_result)
display(experiment.parameter_report())
fig=show_warmup_history(RUN_ROOT);plt.show();plt.close(fig)
if (RUN_ROOT/'warmup_readiness_history.json').exists():
    readiness=json.loads((RUN_ROOT/'warmup_readiness_history.json').read_text())
    display(pd.DataFrame([{'step':x['warmup_step'],'ready':x['ready'],
                           'consecutive':x['consecutive_ready'],**x['flags'],**x['rare_recalls']}
                          for x in readiness]))
if not warmup_result['transition_ready']:
    raise RuntimeError('Warm-up chưa đạt gate; xem WARMUP_BLOCKED.json/readiness history. Không chạy DMI.')
assert (RUN_ROOT/'warmup_transition.pt').exists()


## 7. Smoke test exact DMI sau warm-up


In [ ]:
smoke=experiment.smoke_test()
display(smoke)
assert smoke['effective_batch_size']==16
assert smoke['DMI']['rank']==11 and smoke['DMI']['finite_loss']
assert smoke['gradients']['finite']
assert smoke['parameter_report']['encoder_trainable']==0
assert smoke['parameter_report']['morphology_trainable']==99


## 8. Pilot direct DMI đến step 78

Đây là step nội bộ của giai đoạn DMI, tách biệt với warm-up step. Chạy lại không vượt quá 78. Pilot tạo thử cả hai checkpoint validation; chưa phải kết quả final.


In [ ]:
PILOT_DMI_TARGET_STEP=78
pilot_result=experiment.train(target_step=PILOT_DMI_TARGET_STEP)
display(pilot_result)
display(show_selected_checkpoints(experiment))
fig=show_history(RUN_ROOT);plt.show();plt.close(fig)
if (RUN_ROOT/'dmi_latest_class_permutation.json').exists():
    display(json.loads((RUN_ROOT/'dmi_latest_class_permutation.json').read_text()))
if (RUN_ROOT/'dmi_latest_validation_per_class.csv').exists():
    display(pd.read_csv(RUN_ROOT/'dmi_latest_validation_per_class.csv'))


## 9. Full DMI training — chỉ bật sau khi duyệt pilot


In [ ]:
RUN_FULL_DMI_TRAINING=False
if RUN_FULL_DMI_TRAINING:
    training_result=experiment.train()
    display(training_result)
    display(show_selected_checkpoints(experiment))
    fig=show_history(RUN_ROOT)
    fig.savefig(RUN_ROOT/'dmi_training_curves.png',dpi=160,bbox_inches='tight')
    plt.show();plt.close(fig)
else:
    print('Đang dừng ở DMI pilot. Chỉ đổi RUN_FULL_DMI_TRAINING=True sau khi duyệt diagnostics.')


## 10. Hai checkpoint cuối và model selection


In [ ]:
display(show_selected_checkpoints(experiment))
selection_path=RUN_ROOT/'MODEL_SELECTION.json'
if selection_path.exists():
    display(json.loads(selection_path.read_text()))
else:
    print('MODEL_SELECTION.json chỉ được niêm phong khi DMI training hoàn tất.')


## 11. Xuất bản đồ từ checkpoint macro-F1


In [ ]:
EXPORT_FINAL_MAP=False
if EXPORT_FINAL_MAP:
    map_result=experiment.map()
    display(map_result)
    fig=show_map(experiment)
    fig.savefig(RUN_ROOT/'map_preview.png',dpi=180,bbox_inches='tight')
    plt.show();plt.close(fig)
    print('GeoTIFF:',RUN_ROOT/'map'/'classes_sep21_2026.tif')
else:
    print('Giữ EXPORT_FINAL_MAP=False cho đến khi DMI_TRAINING_COMPLETE.json tồn tại.')


## 12. Tùy chọn: đối chiếu điểm cũ sau khi chốt bản đồ


In [ ]:
EVALUATE_OLD_POINTS=False
POINTS_ROOT=Path('/content/drive/MyDrive/Prithvi_Colab/data')
if EVALUATE_OLD_POINTS:
    from prithvi_xtseg.inference import evaluate_reference_points
    agreement=evaluate_reference_points(experiment.scene,RUN_ROOT,POINTS_ROOT,confirm=True)
    display(agreement)
    display(pd.read_csv(RUN_ROOT/'reference_point_agreement'/'per_class.csv'))
else:
    print('Bỏ qua đối chiếu điểm cũ.')


## Giới hạn diễn giải

- WCE warm-up chỉ là bước hiệu chỉnh task-side ngắn; checkpoint này không phải model cuối.
- Direct DMI vẫn sử dụng nhãn lịch sử; không gọi đây là học không giám sát hay khẳng định không phụ thuộc nhãn.
- Permutation chỉ sửa ánh xạ channel một-một; không sửa mất lớp, nhập lớp hoặc lỗi không gian.
- Validation dùng nhãn lịch sử chưa rõ ngày, nên metric không phải accuracy độc lập của bản đồ tháng 9.
- Smooth logit closing có thể giảm phân mảnh nhưng không bảo đảm tăng accuracy.
- Ảnh chưa có SCL/cloud mask; ảnh 12/08 có dấu hiệu mây.
- Output notebook mới chỉ là cấu hình/code. Chỉ coi số liệu trong run P5 là kết quả mới sau khi các cell thực sự chạy trên Colab.
